上一章我们讲了 Go 并发的三根主线：goroutine、GMP、channel。

这一章继续往工程实践推进，重点解决五类问题：

第一，sync 包。
Java 里常问 synchronized、ReentrantLock、AQS、CountDownLatch、线程池；Go 里对应最常考的是 sync.Mutex、sync.RWMutex、sync.WaitGroup、sync.Once、sync.Cond、sync.Pool、sync.Map。这些工具解决的是共享状态保护、等待一组任务完成、只执行一次、对象复用、并发 map 等问题。

第二，atomic。
Java 里有 CAS、AtomicInteger、AtomicReference、ABA 问题；Go 里对应 sync/atomic 包，包括 atomic.Int64、atomic.Bool、atomic.Value、CompareAndSwap 等。面试重点是 atomic 和 mutex 怎么选、CAS 是什么、原子操作能不能保证复杂逻辑线程安全。

第三，context。
Java 里经常用 ThreadLocal 传递用户信息，用线程池 Future 做取消和超时；Go 里 context 是请求链路中非常核心的东西。它用于取消、超时、deadline、请求级元数据传递，也是防止 goroutine 泄漏的核心工具。

第四，worker pool。
Go 不鼓励像 Java 那样直接照搬线程池，因为 goroutine 很轻量。但工程里仍然需要控制并发度、保护下游资源、处理任务队列、错误传播和优雅关闭，这就引出 worker pool、semaphore、errgroup 等模式。

第五，并发经典题。
面试里经常会让你写两个 goroutine 交替打印、三个 goroutine 顺序打印 ABC、多个 worker 并发处理任务、生产者消费者、超时取消等。这些题不难，但能看出你是不是真的理解 channel、锁、WaitGroup 和 context。

这一章最终要形成一句话：

Go 并发不是只会 `go func()` 和 channel，而是要能根据场景选择 sync、atomic、context、channel、worker pool，控制共享状态、生命周期、并发度和错误传播。

---

# 一、sync.Mutex

## 【文本块 2】Q：sync.Mutex 是什么？解决什么问题？

sync.Mutex 是 Go 标准库提供的互斥锁。

它的作用是保护共享资源，保证同一时刻只有一个 goroutine 能进入临界区。

典型场景：

* 多个 goroutine 同时修改 map
* 多个 goroutine 同时修改计数器
* 多个 goroutine 同时修改结构体字段
* 多个 goroutine 同时访问缓存状态
* 多个 goroutine 同时写共享 buffer

基本用法是：

```go
mu.Lock()
// 临界区
mu.Unlock()
```

工程里一般会配合 defer：

```go
mu.Lock()
defer mu.Unlock()
```

这样即使中间 return，也能释放锁。

面试里可以这样回答：

Mutex 是 Go 里最基础的互斥锁，用来保护共享状态。多个 goroutine 访问同一份可变数据时，如果至少有一个写操作，就要考虑加锁。Mutex 的核心作用是让临界区串行执行，避免数据竞争。

---

## 【代码块 1】不加锁导致数据竞争

```go
package main

import (
    "fmt"
    "sync"
)

func main() {
    count := 0

    var wg sync.WaitGroup

    for i := 0; i < 1000; i++ {
        wg.Add(1)
        go func() {
            defer wg.Done()
            count++
        }()
    }

    wg.Wait()

    fmt.Println(count)
}
```

---

## 【文本块 3】代码解释

这里理论上希望输出 1000，但实际结果可能小于 1000。

原因是 `count++` 不是原子操作，它至少包含三步：

1. 读取 count
2. 加一
3. 写回 count

多个 goroutine 并发执行时，可能都读到同一个旧值，然后互相覆盖写回，导致丢失更新。

这就是典型数据竞争。

---

## 【代码块 2】用 Mutex 保护共享计数器

```go
package main

import (
    "fmt"
    "sync"
)

func main() {
    count := 0

    var mu sync.Mutex
    var wg sync.WaitGroup

    for i := 0; i < 1000; i++ {
        wg.Add(1)
        go func() {
            defer wg.Done()

            mu.Lock()
            count++
            mu.Unlock()
        }()
    }

    wg.Wait()

    fmt.Println(count)
}
```

---

## 【文本块 4】为什么 defer Unlock 更常见？

更推荐写成：

```go
mu.Lock()
defer mu.Unlock()
```

因为这样不容易忘记释放锁。

尤其是临界区里有多个 return 分支时，如果手动写 Unlock，很容易漏掉。

不过在极端热点路径里，defer 有少量额外开销，可以手动 Unlock。但普通业务代码里，优先选择可读性和安全性。

---

## 【代码块 3】defer Unlock 写法

```go
package main

import (
    "fmt"
    "sync"
)

type Counter struct {
    mu    sync.Mutex
    count int
}

func (c *Counter) Inc() {
    c.mu.Lock()
    defer c.mu.Unlock()

    c.count++
}

func (c *Counter) Value() int {
    c.mu.Lock()
    defer c.mu.Unlock()

    return c.count
}

func main() {
    c := &Counter{}

    var wg sync.WaitGroup

    for i := 0; i < 1000; i++ {
        wg.Add(1)
        go func() {
            defer wg.Done()
            c.Inc()
        }()
    }

    wg.Wait()

    fmt.Println(c.Value())
}
```

---

## 【文本块 5】Q：Mutex 可以复制吗？

不能复制已经使用过的 Mutex。

sync.Mutex 内部有锁状态。如果复制一个已经使用过的 Mutex，相当于复制了锁状态，但两个 Mutex 后续又不是同一把锁，很容易导致并发保护失效或死锁。

所以包含 Mutex 的结构体也不要随便复制。

错误风险：

```go
type SafeCounter struct {
    mu sync.Mutex
    n  int
}
```

如果把 SafeCounter 按值传递，就会复制里面的 mu。

工程建议：

* 含 Mutex 的结构体一般用指针传递
* 方法接收者用 pointer receiver
* 不要把含锁对象放进会复制的容器逻辑里
* 可以用 go vet 检查 copylocks 问题

面试里可以这样回答：

Mutex 一旦使用后不能复制。因为 Mutex 内部有状态，复制会导致锁状态和被保护数据的关系被破坏。包含 Mutex 的结构体通常应该通过指针传递，方法也用指针接收者。

---

# 二、sync.RWMutex

## 【文本块 6】Q：RWMutex 是什么？和 Mutex 有什么区别？

sync.RWMutex 是读写锁。

它把锁分成两类：

* 读锁：RLock / RUnlock
* 写锁：Lock / Unlock

规则是：

1. 多个读锁可以同时持有。
2. 写锁和读锁互斥。
3. 写锁和写锁互斥。

所以 RWMutex 适合读多写少的场景。

比如：

* 配置缓存
* 本地只读多写少 map
* 路由表
* 元数据缓存
* 读多写少的内存索引

如果读写都很多，RWMutex 不一定比 Mutex 好。因为 RWMutex 维护读写状态也有成本，而且写锁需要等待已有读锁释放。

面试里可以这样回答：

Mutex 是完全互斥锁，读写都互斥；RWMutex 允许多个读并发，但写和读、写和写都互斥。它适合读多写少的共享状态，如果写很多或者临界区很短，RWMutex 不一定比 Mutex 快。

---

## 【代码块 4】RWMutex 保护读多写少 map

```go
package main

import (
    "fmt"
    "sync"
)

type Cache struct {
    mu sync.RWMutex
    m  map[string]string
}

func NewCache() *Cache {
    return &Cache{
        m: make(map[string]string),
    }
}

func (c *Cache) Set(key, value string) {
    c.mu.Lock()
    defer c.mu.Unlock()

    c.m[key] = value
}

func (c *Cache) Get(key string) (string, bool) {
    c.mu.RLock()
    defer c.mu.RUnlock()

    v, ok := c.m[key]
    return v, ok
}

func main() {
    cache := NewCache()

    cache.Set("name", "go")

    v, ok := cache.Get("name")
    fmt.Println(v, ok)
}
```

---

## 【文本块 7】追问：RWMutex 一定比 Mutex 性能好吗？

不一定。

RWMutex 的优势建立在读多写少的前提上。

如果读很多、写很少，多个读 goroutine 可以并发执行，RWMutex 通常有优势。

但如果写操作频繁，写锁会阻塞所有读锁，同时还要等待已有读锁释放，性能可能不如普通 Mutex。

如果临界区非常短，RWMutex 的额外状态维护成本也可能抵消收益。

所以工程里更稳妥的说法是：

> 读多写少且读临界区有一定耗时时，用 RWMutex；竞争不明显或者读写比例不确定时，先用 Mutex，必要时通过 benchmark 或压测验证。

---

## 【文本块 8】Q：RWMutex 可以从读锁升级成写锁吗？

不能安全升级。

错误想法：

```go
mu.RLock()
if needWrite {
    mu.Lock()
}
```

这样很容易死锁。

因为当前 goroutine 持有读锁时，再申请写锁，写锁要等待所有读锁释放，其中也包括自己持有的读锁。

正确做法是：

1. 先释放读锁
2. 再获取写锁
3. 获取写锁后重新检查条件

这叫 double-check。

---

## 【代码块 5】读锁释放后再获取写锁

```go
package main

import (
    "fmt"
    "sync"
)

type Store struct {
    mu sync.RWMutex
    m  map[string]string
}

func NewStore() *Store {
    return &Store{
        m: make(map[string]string),
    }
}

func (s *Store) GetOrSet(key, value string) string {
    s.mu.RLock()
    v, ok := s.m[key]
    s.mu.RUnlock()

    if ok {
        return v
    }

    s.mu.Lock()
    defer s.mu.Unlock()

    // 重新检查，防止其他 goroutine 已经写入
    if v, ok := s.m[key]; ok {
        return v
    }

    s.m[key] = value
    return value
}

func main() {
    s := NewStore()
    fmt.Println(s.GetOrSet("lang", "go"))
}
```

---

# 三、sync.WaitGroup

## 【文本块 9】Q：WaitGroup 是什么？解决什么问题？

sync.WaitGroup 用来等待一组 goroutine 执行完成。

它有三个核心方法：

```go
Add(delta int)
Done()
Wait()
```

典型用法：

1. 启动 goroutine 前调用 Add。
2. goroutine 结束时调用 Done。
3. 主 goroutine 调用 Wait 等待所有任务完成。

WaitGroup 很像 Java 里的 CountDownLatch，但更轻量，使用也更直接。

面试里可以这样回答：

WaitGroup 用来等待一组 goroutine 完成。Add 增加计数，Done 减少计数，Wait 阻塞等待计数归零。它只负责等待，不负责取消、错误传播和结果收集。

---

## 【代码块 6】WaitGroup 基本使用

```go
package main

import (
    "fmt"
    "sync"
    "time"
)

func main() {
    var wg sync.WaitGroup

    for i := 1; i <= 3; i++ {
        i := i

        wg.Add(1)
        go func() {
            defer wg.Done()

            time.Sleep(100 * time.Millisecond)
            fmt.Println("task done:", i)
        }()
    }

    wg.Wait()

    fmt.Println("all done")
}
```

---

## 【文本块 10】为什么 Add 要在 goroutine 启动前调用？

正确写法：

```go
wg.Add(1)
go func() {
    defer wg.Done()
}()
```

错误写法：

```go
go func() {
    wg.Add(1)
    defer wg.Done()
}()
wg.Wait()
```

如果 Add 写在 goroutine 里面，主 goroutine 可能先执行到 Wait。此时计数还是 0，Wait 直接返回。随后子 goroutine 才 Add，这就破坏了等待语义。

所以面试里要强调：

> WaitGroup 的 Add 应该在启动 goroutine 之前调用，避免 Wait 先看到计数为 0 而提前返回。

---

## 【代码块 7】错误示例：不要把 Add 放进 goroutine

```go
package main

import (
    "fmt"
    "sync"
    "time"
)

func main() {
    var wg sync.WaitGroup

    for i := 1; i <= 3; i++ {
        i := i

        go func() {
            wg.Add(1)
            defer wg.Done()

            time.Sleep(100 * time.Millisecond)
            fmt.Println("task done:", i)
        }()
    }

    wg.Wait()
    fmt.Println("main may exit early")
}
```

---

## 【文本块 11】WaitGroup 常见坑

第一，Done 调用次数不能超过 Add。
否则计数会变成负数，直接 panic。

第二，Add 最好在启动 goroutine 前调用。
避免 Wait 提前返回。

第三，WaitGroup 不能复制。
它内部有状态，复制后会破坏同步语义。

第四，WaitGroup 不处理 error。
如果 goroutine 里出错，WaitGroup 只知道它结束了，不知道成功还是失败。

第五，WaitGroup 不负责取消。
如果某个 goroutine 卡住，Wait 会一直等。要配合 context 控制生命周期。

工程里如果需要“等待 + 错误传播 + 出错取消其他任务”，更推荐 errgroup。

---

# 四、sync.Once

## 【文本块 12】Q：sync.Once 是什么？有什么用？

sync.Once 用来保证某段逻辑在并发环境下只执行一次。

典型场景：

* 单例初始化
* 加载配置
* 初始化连接池
* 编译正则
* 初始化全局缓存
* 惰性加载资源

基本用法：

```go
var once sync.Once

once.Do(func() {
    // 只执行一次
})
```

即使多个 goroutine 同时调用 Do，里面的函数也只会执行一次。

面试里可以这样回答：

sync.Once 用来做并发安全的一次性初始化。它保证传入 Do 的函数全局只执行一次，即使多个 goroutine 同时调用也不会重复执行。常用于单例、配置加载、资源初始化。

---

## 【代码块 8】sync.Once 基本使用

```go
package main

import (
    "fmt"
    "sync"
)

type Config struct {
    Name string
}

var (
    config *Config
    once   sync.Once
)

func GetConfig() *Config {
    once.Do(func() {
        fmt.Println("init config")
        config = &Config{Name: "app"}
    })

    return config
}

func main() {
    var wg sync.WaitGroup

    for i := 0; i < 5; i++ {
        wg.Add(1)

        go func() {
            defer wg.Done()
            fmt.Println(GetConfig().Name)
        }()
    }

    wg.Wait()
}
```

---

## 【文本块 13】sync.Once 的底层原理怎么理解？

sync.Once 内部大致有两个关键点：

1. 一个 done 标记，表示是否执行过。
2. 一个 Mutex，用来保证并发下只有一个 goroutine 真正执行初始化函数。

常见优化逻辑是：

先用原子读快速判断 done。
如果已经 done，就直接返回，避免加锁。
如果没 done，再加锁，二次检查，然后执行函数，最后设置 done。

这和单例模式里的双重检查有点类似。

面试里可以这样回答：

Once 不是简单每次都加锁。它通常会先用原子标记做快速路径，没初始化时再加锁执行，执行完成后设置 done。这样初始化后再次调用 Do 的成本很低。

---

## 【文本块 14】追问：Once 里的函数 panic 了，下次还会执行吗？

不会。

sync.Once 的语义是：Do 被调用一次就认为执行过。即使传入函数发生 panic，Once 也会认为这次 Do 已经完成，后续不会再次执行。

这点非常重要。

如果初始化函数可能失败，并且希望失败后重试，sync.Once 不是直接合适的工具，需要自己封装状态。

---

## 【代码块 9】Once 函数 panic 后不会重试

```go
package main

import (
    "fmt"
    "sync"
)

func main() {
    var once sync.Once

    for i := 0; i < 2; i++ {
        func() {
            defer func() {
                if r := recover(); r != nil {
                    fmt.Println("recover:", r)
                }
            }()

            once.Do(func() {
                fmt.Println("run once")
                panic("init failed")
            })
        }()
    }

    fmt.Println("done")
}
```

---

# 五、sync.Cond

## 【文本块 15】Q：sync.Cond 是什么？什么时候用？

sync.Cond 是条件变量，用于多个 goroutine 等待某个条件成立。

它通常和 Mutex 搭配使用。

基本方法：

```go
Wait()
Signal()
Broadcast()
```

Wait 表示等待条件。
Signal 唤醒一个等待者。
Broadcast 唤醒所有等待者。

sync.Cond 的使用场景没有 channel 那么常见，但在一些复杂等待条件下很有用。

例如：

* 多个 goroutine 等待队列非空
* 多个 goroutine 等待状态变化
* 多个消费者等待生产者生产数据
* 一个事件发生后唤醒所有等待者

面试里可以这样回答：

sync.Cond 是条件变量，用来让 goroutine 等待某个条件成立。它适合复杂共享状态条件等待。简单通知可以用 channel，但如果条件和共享状态绑定很紧，Cond 会更自然。

---

## 【代码块 10】sync.Cond 示例：等待队列非空

```go
package main

import (
    "fmt"
    "sync"
    "time"
)

func main() {
    var mu sync.Mutex
    cond := sync.NewCond(&mu)

    queue := make([]int, 0)

    go func() {
        mu.Lock()
        for len(queue) == 0 {
            cond.Wait()
        }

        item := queue[0]
        queue = queue[1:]
        mu.Unlock()

        fmt.Println("consume:", item)
    }()

    time.Sleep(100 * time.Millisecond)

    mu.Lock()
    queue = append(queue, 1)
    cond.Signal()
    mu.Unlock()

    time.Sleep(100 * time.Millisecond)
}
```

---

## 【文本块 16】为什么 Wait 要放在 for 循环里？

Cond 的 Wait 必须放在 for 循环里检查条件：

```go
for !condition {
    cond.Wait()
}
```

原因有两个：

第一，可能被虚假唤醒或被其他事件唤醒。
第二，多个 goroutine 被唤醒后，条件可能已经被别的 goroutine 消费掉了。

所以不能写：

```go
if !condition {
    cond.Wait()
}
```

这和 Java 里 wait/notify 也很类似：被唤醒后要重新检查条件。

---

## 【文本块 17】Cond 和 channel 怎么选？

简单一次性通知，用 channel 更简单。

例如：

```go
done := make(chan struct{})
close(done)
```

复杂条件等待，用 Cond 更合适。

例如：

* 等待队列非空
* 等待容量不满
* 多生产者多消费者
* 同一个条件会反复变真变假
* 需要 Broadcast 唤醒多个等待者

面试回答：

channel 更适合传递数据和一次性事件通知；Cond 更适合围绕共享状态做复杂条件等待，尤其是条件会反复变化，需要多个 goroutine 等待和唤醒的场景。

---

# 六、sync.Pool

## 【文本块 18】Q：sync.Pool 是什么？有什么用？

sync.Pool 是 Go 提供的临时对象复用池。

它适合复用短生命周期、创建成本较高、并且可以安全重置的临时对象。

典型场景：

* bytes.Buffer
* 临时 []byte
* 编码解码临时对象
* 日志 buffer
* 高频请求中的临时结构

sync.Pool 的目标是减少频繁分配对象带来的 GC 压力。

基本用法：

```go
pool := sync.Pool{
    New: func() any {
        return new(bytes.Buffer)
    },
}
```

获取对象：

```go
buf := pool.Get().(*bytes.Buffer)
```

归还对象：

```go
buf.Reset()
pool.Put(buf)
```

面试里可以这样回答：

sync.Pool 是临时对象复用池，用来减少频繁分配和 GC 压力。它适合缓存无状态、可重置的临时对象，比如 bytes.Buffer。但 Pool 里的对象可能随时被 GC 清掉，所以不能把它当成连接池或对象生命周期管理工具。

---

## 【代码块 11】sync.Pool 复用 bytes.Buffer

```go
package main

import (
    "bytes"
    "fmt"
    "sync"
)

func main() {
    var pool = sync.Pool{
        New: func() any {
            return new(bytes.Buffer)
        },
    }

    buf := pool.Get().(*bytes.Buffer)
    buf.Reset()

    buf.WriteString("hello")
    buf.WriteString(" pool")

    fmt.Println(buf.String())

    buf.Reset()
    pool.Put(buf)
}
```

---

## 【文本块 19】sync.Pool 的注意事项

第一，Put 前要重置对象。
例如 bytes.Buffer 要 Reset，否则下次拿到会有脏数据。

第二，Pool 里的对象可能被 GC 清掉。
所以不能依赖 Pool 一定保存对象。

第三，不要用 sync.Pool 做数据库连接池、TCP 连接池。
连接池需要明确生命周期、健康检查、最大连接数、关闭逻辑；sync.Pool 不提供这些保证。

第四，Pool 适合临时对象，不适合有状态业务对象。
如果对象带用户信息、请求上下文、事务状态，放进 Pool 很危险。

第五，小对象不一定值得 Pool。
如果对象创建很便宜，Pool 反而增加复杂度。

---

# 七、sync.Map

## 【文本块 20】Q：sync.Map 是什么？适合什么场景？

sync.Map 是 Go 标准库提供的并发安全 map。

它适合两类场景：

第一，写入一次，读取很多次。
比如只增不改的缓存、注册表。

第二，多个 goroutine 操作不同 key，key 相对稳定。
比如连接表、全局元数据。

sync.Map 不一定适合频繁写、频繁删除、key 变化很大的场景。普通 map + Mutex/RWMutex 往往更清晰，也可能更快。

sync.Map 常用方法：

```go
Store(key, value)
Load(key)
LoadOrStore(key, value)
Delete(key)
Range(func(key, value any) bool)
```

面试里可以这样回答：

sync.Map 不是普通 map 加锁的万能替代品。它针对读多写少、key 稳定的场景做了优化。普通业务里，如果读写逻辑明确，我会优先使用 map + Mutex/RWMutex；只有在读多写少且 key 稳定时考虑 sync.Map。

---

## 【代码块 12】sync.Map 使用示例

```go
package main

import (
    "fmt"
    "sync"
)

func main() {
    var m sync.Map

    m.Store("a", 1)
    m.Store("b", 2)

    if v, ok := m.Load("a"); ok {
        fmt.Println("a:", v)
    }

    actual, loaded := m.LoadOrStore("c", 3)
    fmt.Println("c:", actual, "loaded:", loaded)

    m.Range(func(key, value any) bool {
        fmt.Println(key, value)
        return true
    })
}
```

---

# 八、atomic 基础

## 【文本块 21】Q：atomic 是什么？解决什么问题？

atomic 指原子操作。

它保证某个单变量操作不可被中断，避免并发读写时出现数据竞争。

Go 的原子操作在 sync/atomic 包里。

常见类型：

```go
atomic.Int64
atomic.Int32
atomic.Uint64
atomic.Uint32
atomic.Bool
atomic.Pointer[T]
atomic.Value
```

常见操作：

```go
Load
Store
Add
Swap
CompareAndSwap
```

atomic 适合非常简单的共享状态，比如：

* 计数器
* 开关标志
* 状态位
* 统计指标
* 读多写少的配置快照
* 无锁更新单个变量

面试里可以这样回答：

atomic 用来保证单个变量操作的原子性和可见性。它比 Mutex 更轻量，但只适合简单状态。复杂不变式、多字段一致性、复合逻辑还是应该用锁。

---

## 【代码块 13】atomic.Int64 计数器

```go
package main

import (
    "fmt"
    "sync"
    "sync/atomic"
)

func main() {
    var count atomic.Int64

    var wg sync.WaitGroup

    for i := 0; i < 1000; i++ {
        wg.Add(1)

        go func() {
            defer wg.Done()
            count.Add(1)
        }()
    }

    wg.Wait()

    fmt.Println(count.Load())
}
```

---

## 【文本块 22】atomic 和 Mutex 怎么选？

如果只是单个变量的简单读写或递增，用 atomic 很合适。

例如：

```go
counter.Add(1)
flag.Store(true)
```

如果是多个变量之间要保持一致性，用 Mutex。

例如：

```go
balance -= amount
order.Status = Paid
log.Append(...)
```

这些操作必须整体一致，不能只保证某个字段原子。

如果逻辑里有“先检查再修改”，也通常需要锁。因为检查和修改之间可能被其他 goroutine 插入。

面试回答：

atomic 适合简单单变量状态，Mutex 适合复杂临界区和多字段不变式。atomic 性能更轻，但可读性和组合能力差；Mutex 语义更清晰，适合大多数业务共享状态。

---

# 九、CAS

## 【文本块 23】Q：CAS 是什么？

CAS 全称 Compare And Swap，也就是比较并交换。

它包含三个值：

* 当前内存值 V
* 期望值 old
* 新值 new

执行逻辑：

如果 V 等于 old，就把 V 更新为 new，并返回成功。
如果 V 不等于 old，说明被别的 goroutine 改过，本次更新失败。

CAS 是一种乐观并发控制思想。
它不先加锁，而是在提交修改时检查有没有冲突。

Go 里 CAS 方法通常叫：

```go
CompareAndSwap
```

例如：

```go
flag.CompareAndSwap(false, true)
```

面试里可以这样回答：

CAS 是比较并交换，它先判断变量当前值是否等于预期值，如果相等就更新，否则失败。它是一种无锁的乐观并发控制，适合低竞争、短操作的场景。

---

## 【代码块 14】atomic.CompareAndSwap 示例

```go
package main

import (
    "fmt"
    "sync/atomic"
)

func main() {
    var flag atomic.Bool

    ok := flag.CompareAndSwap(false, true)

    fmt.Println("swap success:", ok)
    fmt.Println("flag:", flag.Load())

    ok = flag.CompareAndSwap(false, true)

    fmt.Println("swap success:", ok)
    fmt.Println("flag:", flag.Load())
}
```

---

## 【文本块 24】CAS 的优缺点

优点：

1. 不需要阻塞 goroutine。
2. 低竞争下性能好。
3. 适合简单状态更新。
4. 可以避免锁带来的上下文切换和阻塞。

缺点：

1. 高竞争下可能反复失败，自旋浪费 CPU。
2. 只能自然表达单变量更新。
3. 复杂逻辑难写，容易出 bug。
4. 可能有 ABA 问题。
5. 可读性不如锁。

所以 CAS 不是锁的万能替代品。

面试里可以这样总结：

CAS 适合低竞争、单变量、短逻辑场景。高竞争下 CAS 会不断失败重试，浪费 CPU；复杂业务状态还是用 Mutex 更清晰可靠。

---

## 【文本块 25】Q：什么是 ABA 问题？

ABA 问题指的是：

一个 goroutine 看到变量值是 A。
期间另一个 goroutine 把它从 A 改成 B，又从 B 改回 A。
第一个 goroutine 再做 CAS 时发现还是 A，于是认为没变过，CAS 成功。

但实际上变量已经经历过变化。

在一些只关心最终值的场景，ABA 不一定有害。
但在链表、栈、对象复用、指针 CAS 等场景，ABA 可能导致严重问题。

常见解决方案：

* 加版本号
* 加时间戳
* 使用指针加 tag
* 用锁简化逻辑

Go 标准 atomic 没有像 Java AtomicStampedReference 那样直接提供版本引用类型，但可以自己把值和版本封装起来，或者避免用 CAS 实现复杂无锁结构。

---

# 十、atomic.Value

## 【文本块 26】Q：atomic.Value 有什么用？

atomic.Value 用来原子地存取任意类型的值。

典型场景是读多写少的配置热更新。

比如：

* 配置快照
* 路由表快照
* 黑白名单
* 策略规则
* 限流规则

写入时构造一个新的完整对象，然后 Store。
读取时直接 Load 当前快照，不需要加锁。

这是一种 copy-on-write 思路。

面试里可以这样回答：

atomic.Value 适合存储读多写少的整体快照。写的时候构造新对象并原子替换，读的时候无锁读取当前快照。它适合配置、规则表、路由表这类场景。

---

## 【代码块 15】atomic.Value 配置快照

```go
package main

import (
    "fmt"
    "sync/atomic"
)

type Config struct {
    Version int
    Name    string
}

func main() {
    var v atomic.Value

    v.Store(&Config{
        Version: 1,
        Name:    "old",
    })

    cfg := v.Load().(*Config)
    fmt.Println(cfg.Version, cfg.Name)

    v.Store(&Config{
        Version: 2,
        Name:    "new",
    })

    cfg = v.Load().(*Config)
    fmt.Println(cfg.Version, cfg.Name)
}
```

---

## 【文本块 27】atomic.Value 注意事项

第一，不能 Load 一个从未 Store 的 atomic.Value。
否则会返回 nil，类型断言时可能 panic。

第二，Store 的具体类型必须一致。
第一次 Store 了 `*Config`，后面也必须 Store `*Config`，不能换成 Config 或其他类型。

第三，不要 Store 后继续修改对象内部字段。
否则读 goroutine 可能看到并发修改。正确做法是构造新对象，然后整体 Store。

第四，atomic.Value 适合替换整体快照，不适合频繁修改内部字段。

---

# 十一、context 基础

## 【文本块 28】Q：context 是什么？解决什么问题？

context 是 Go 里用于跨 API 边界传递取消信号、超时时间、deadline 和请求级元数据的标准方式。

它主要解决四个问题：

第一，取消。
上游请求取消后，下游 goroutine、数据库查询、RPC 调用也应该停止。

第二，超时。
一个请求不能无限等待数据库、Redis、第三方接口。

第三，deadline。
规定某个时间点之前必须完成。

第四，请求级值传递。
比如 trace_id、request_id、用户 ID、鉴权信息等轻量元数据。

context 最常见的类型：

```go
context.Background()
context.TODO()
context.WithCancel()
context.WithTimeout()
context.WithDeadline()
context.WithValue()
```

面试里可以这样回答：

context 是 Go 中管理请求生命周期的核心工具，用来传递取消、超时、deadline 和请求级元数据。它能让一个请求链路中的 goroutine、数据库、RPC、HTTP 调用在上游取消或超时时及时退出，避免资源泄漏。

---

## 【代码块 16】context.WithCancel

```go
package main

import (
    "context"
    "fmt"
    "time"
)

func worker(ctx context.Context) {
    for {
        select {
        case <-ctx.Done():
            fmt.Println("worker canceled:", ctx.Err())
            return
        default:
            fmt.Println("working")
            time.Sleep(100 * time.Millisecond)
        }
    }
}

func main() {
    ctx, cancel := context.WithCancel(context.Background())

    go worker(ctx)

    time.Sleep(300 * time.Millisecond)
    cancel()

    time.Sleep(100 * time.Millisecond)
}
```

---

## 【文本块 29】context.WithTimeout 和 WithDeadline 区别

WithTimeout 表示从现在开始过多久超时。

```go
ctx, cancel := context.WithTimeout(parent, 2*time.Second)
```

WithDeadline 表示到某个具体时间点超时。

```go
ctx, cancel := context.WithDeadline(parent, time.Now().Add(2*time.Second))
```

本质上 WithTimeout 是 WithDeadline 的简化形式。

常见场景：

* HTTP 请求整体超时
* 数据库查询超时
* Redis 操作超时
* RPC 调用超时
* 批处理任务超时
* goroutine 生命周期控制

---

## 【代码块 17】context.WithTimeout

```go
package main

import (
    "context"
    "fmt"
    "time"
)

func main() {
    ctx, cancel := context.WithTimeout(context.Background(), 200*time.Millisecond)
    defer cancel()

    select {
    case <-time.After(500 * time.Millisecond):
        fmt.Println("task done")
    case <-ctx.Done():
        fmt.Println("timeout:", ctx.Err())
    }
}
```

---

## 【文本块 30】为什么 cancel 一定要调用？

即使 context 会因为 timeout 自动取消，也推荐调用 cancel。

原因是 cancel 可以提前释放 context 相关资源，比如 timer。

标准写法：

```go
ctx, cancel := context.WithTimeout(parent, time.Second)
defer cancel()
```

如果不调用 cancel，相关资源要等到超时或父 context 取消才释放。

面试里可以这样回答：

WithCancel、WithTimeout、WithDeadline 返回的 cancel 应该调用，通常 defer cancel。这样可以及时释放 timer 和取消信号相关资源，也能让下游尽快退出。

---

# 十二、context.Value

## 【文本块 31】Q：context.Value 能传什么？能不能乱用？

context.Value 用于传递请求范围内的元数据。

适合传：

* trace_id
* request_id
* 用户身份
* 租户 ID
* 语言环境
* 链路追踪字段
* 鉴权信息摘要

不适合传：

* 必需业务参数
* 大对象
* 可变对象
* 数据库连接
* logger 的复杂全局状态
* 可选但很重的业务对象

最重要的原则：

> context.Value 只传请求级元数据，不要用它替代函数参数。

如果某个参数是业务逻辑必需的，就应该显式写在函数参数里，而不是藏在 context 里。

面试里可以这样回答：

context.Value 适合传跨层的请求元数据，比如 trace_id、user_id，不适合传业务必需参数和大对象。滥用 context.Value 会让依赖关系隐藏起来，代码可读性和可测试性变差。

---

## 【代码块 18】context.Value 示例

```go
package main

import (
    "context"
    "fmt"
)

type contextKey string

const traceIDKey contextKey = "trace_id"

func logWithTrace(ctx context.Context, msg string) {
    traceID, _ := ctx.Value(traceIDKey).(string)
    fmt.Println("trace_id:", traceID, "msg:", msg)
}

func main() {
    ctx := context.WithValue(context.Background(), traceIDKey, "abc-123")

    logWithTrace(ctx, "hello")
}
```

---

## 【文本块 32】为什么 context key 不建议用 string？

如果直接用 string 作为 key，不同包之间可能 key 冲突。

例如两个包都用：

```go
ctx.Value("user_id")
```

就可能互相覆盖或误取。

更推荐定义私有类型：

```go
type contextKey string
```

或者：

```go
type userIDKey struct{}
```

这样可以避免跨包冲突。

---

# 十三、context 使用规范

## 【文本块 33】Q：context 为什么通常作为第一个参数？

Go 习惯把 context.Context 作为函数第一个参数：

```go
func QueryUser(ctx context.Context, id int64) (*User, error)
```

原因是：

第一，它是请求生命周期控制参数。
放在第一个位置，调用方一眼能看到这个函数支持取消和超时。

第二，它应该显式传递。
不要把 context 存到结构体里作为长期字段。

第三，它贯穿整个调用链。
HTTP handler、service、dao、rpc client 都应该接受并传递 ctx。

工程规范：

* context 作为第一个参数
* 参数名通常叫 ctx
* 不要传 nil context
* 不要把 context 存进 struct
* 不要滥用 context.Value
* 派生 context 后记得 cancel
* goroutine 内部要监听 ctx.Done

---

## 【代码块 19】context 贯穿调用链

```go
package main

import (
    "context"
    "fmt"
    "time"
)

func QueryDB(ctx context.Context, userID int64) error {
    select {
    case <-time.After(500 * time.Millisecond):
        fmt.Println("query success:", userID)
        return nil
    case <-ctx.Done():
        return ctx.Err()
    }
}

func GetUser(ctx context.Context, userID int64) error {
    return QueryDB(ctx, userID)
}

func main() {
    ctx, cancel := context.WithTimeout(context.Background(), 100*time.Millisecond)
    defer cancel()

    err := GetUser(ctx, 1)
    fmt.Println("err:", err)
}
```

---

# 十四、context 和 goroutine 泄漏

## 【文本块 34】Q：context 如何防止 goroutine 泄漏？

goroutine 泄漏常见原因是：goroutine 启动后一直等待某个事件，但上游已经不需要它了。

context 可以把取消信号传给 goroutine。

goroutine 内部通过：

```go
select {
case <-ctx.Done():
    return
}
```

监听取消。

一旦请求超时、客户端断开、主任务失败，就可以 cancel，所有监听这个 ctx 的 goroutine 都能退出。

这就是 context 在 Go 并发里的核心价值：管理生命周期。

---

## 【代码块 20】context 控制多个 goroutine 退出

```go
package main

import (
    "context"
    "fmt"
    "sync"
    "time"
)

func worker(ctx context.Context, id int, wg *sync.WaitGroup) {
    defer wg.Done()

    for {
        select {
        case <-ctx.Done():
            fmt.Println("worker exit:", id)
            return
        default:
            fmt.Println("worker running:", id)
            time.Sleep(100 * time.Millisecond)
        }
    }
}

func main() {
    ctx, cancel := context.WithCancel(context.Background())

    var wg sync.WaitGroup

    for i := 1; i <= 3; i++ {
        wg.Add(1)
        go worker(ctx, i, &wg)
    }

    time.Sleep(300 * time.Millisecond)
    cancel()

    wg.Wait()
    fmt.Println("all workers exited")
}
```

---

# 十五、worker pool 基础

## 【文本块 35】Q：Go 需要线程池吗？

严格来说，Go 通常不需要像 Java 那样用线程池复用线程。

因为 goroutine 创建成本很低，Go runtime 已经负责把 goroutine 调度到 OS thread 上执行。

但这不代表 Go 不需要控制并发。

工程里真正要控制的是：

* 同时访问数据库的任务数
* 同时请求第三方接口的任务数
* 同时处理文件的任务数
* CPU 密集型任务并发度
* 队列堆积量
* 下游系统承压能力
* 内存占用

所以 Go 里常说不需要“线程池”，但需要“worker pool”或“并发限制”。

面试里可以这样回答：

Go 不需要为了复用线程而设计 Java 式线程池，因为 goroutine 很轻量，runtime 已经做了线程调度。但工程里仍然需要 worker pool 来限制并发度、保护下游资源、控制任务队列和实现背压。

---

## 【文本块 36】worker pool 的基本结构

一个典型 worker pool 包含：

1. jobs channel：任务队列
2. workers：固定数量 goroutine
3. results channel：结果队列，可选
4. WaitGroup：等待 worker 退出
5. context：取消和超时
6. error channel：错误传播，可选
7. close 机制：关闭任务队列，等待消费者退出

核心思路：

主 goroutine 往 jobs 投递任务。
worker goroutine 从 jobs 读取任务并处理。
jobs 关闭后，worker range 结束并退出。
主 goroutine 等待所有 worker 完成。

---

## 【代码块 21】最基本 worker pool

```go
package main

import (
    "fmt"
    "sync"
    "time"
)

type Job struct {
    ID int
}

type Result struct {
    JobID int
    Value int
}

func worker(id int, jobs <-chan Job, results chan<- Result, wg *sync.WaitGroup) {
    defer wg.Done()

    for job := range jobs {
        fmt.Println("worker", id, "processing job", job.ID)

        time.Sleep(100 * time.Millisecond)

        results <- Result{
            JobID: job.ID,
            Value: job.ID * 2,
        }
    }
}

func main() {
    jobs := make(chan Job, 5)
    results := make(chan Result, 5)

    var wg sync.WaitGroup

    workerCount := 3
    jobCount := 5

    for i := 1; i <= workerCount; i++ {
        wg.Add(1)
        go worker(i, jobs, results, &wg)
    }

    for j := 1; j <= jobCount; j++ {
        jobs <- Job{ID: j}
    }
    close(jobs)

    go func() {
        wg.Wait()
        close(results)
    }()

    for result := range results {
        fmt.Println("result:", result)
    }
}
```

---

## 【文本块 37】代码解释

jobs 是任务队列。
results 是结果队列。
启动 3 个 worker 并发处理任务。
主 goroutine 投递 5 个任务后关闭 jobs。
worker 通过 `for job := range jobs` 消费任务。
jobs 关闭且读完后，worker 自动退出。
另一个 goroutine 等待所有 worker 退出后关闭 results。
主 goroutine range results 收集结果。

注意：results 必须在所有 worker 都结束后关闭，否则 worker 还在写 results 时关闭会 panic。

---

# 十六、worker pool 加 context

## 【文本块 38】Q：worker pool 如何支持取消？

真实项目里的 worker pool 不能只处理正常任务，还要支持取消。

比如：

* 请求超时
* 用户取消
* 上游任务失败
* 服务准备关闭
* 达到最大处理时间

这时要把 context 传给 worker。

worker 在两个地方监听 ctx：

1. 等待任务时
2. 处理任务时

等待任务时：

```go
select {
case job, ok := <-jobs:
case <-ctx.Done():
    return
}
```

处理任务时，如果任务本身支持 ctx，也要向下传递。

---

## 【代码块 22】支持取消的 worker pool

```go
package main

import (
    "context"
    "fmt"
    "sync"
    "time"
)

type Job struct {
    ID int
}

func worker(ctx context.Context, id int, jobs <-chan Job, wg *sync.WaitGroup) {
    defer wg.Done()

    for {
        select {
        case <-ctx.Done():
            fmt.Println("worker canceled:", id)
            return

        case job, ok := <-jobs:
            if !ok {
                fmt.Println("worker jobs closed:", id)
                return
            }

            fmt.Println("worker", id, "processing job", job.ID)

            select {
            case <-time.After(200 * time.Millisecond):
                fmt.Println("worker", id, "done job", job.ID)
            case <-ctx.Done():
                fmt.Println("worker", id, "canceled during job", job.ID)
                return
            }
        }
    }
}

func main() {
    ctx, cancel := context.WithTimeout(context.Background(), 500*time.Millisecond)
    defer cancel()

    jobs := make(chan Job)

    var wg sync.WaitGroup

    for i := 1; i <= 3; i++ {
        wg.Add(1)
        go worker(ctx, i, jobs, &wg)
    }

    go func() {
        defer close(jobs)

        for j := 1; j <= 10; j++ {
            select {
            case jobs <- Job{ID: j}:
            case <-ctx.Done():
                return
            }
        }
    }()

    wg.Wait()
    fmt.Println("pool stopped")
}
```

---

## 【文本块 39】worker pool 设计注意事项

第一，任务队列要有边界。
无界队列容易导致内存堆积。Go channel 天然是有界或无缓冲的，更容易做背压。

第二，worker 数量要可配置。
不要写死，根据 CPU、IO、下游能力、压测结果调整。

第三，任务处理要支持 context。
否则取消只能停止取新任务，不能停止正在执行的任务。

第四，错误要有传播机制。
WaitGroup 只等待，不收集错误。

第五，关闭顺序要清楚。
一般是关闭 jobs，让 worker 自然退出；等 worker 全部退出后再关闭 results。

第六，worker 内部要 recover。
避免某个任务 panic 导致整个进程崩溃。

---

# 十七、semaphore 控制并发

## 【文本块 40】Q：不用完整 worker pool，如何简单控制并发数？

可以用有缓冲 channel 实现 semaphore。

例如：

```go
sem := make(chan struct{}, 10)
```

每个任务开始前获取令牌：

```go
sem <- struct{}{}
```

结束后释放令牌：

```go
<-sem
```

容量是 10，就代表最多 10 个任务同时执行。

这个方式适合：

* 批量请求接口
* 批量处理文件
* 批量发送消息
* 控制数据库并发
* 控制 CPU 密集任务并行度

---

## 【代码块 23】channel semaphore 控制并发

```go
package main

import (
    "fmt"
    "sync"
    "time"
)

func main() {
    sem := make(chan struct{}, 3)

    var wg sync.WaitGroup

    for i := 1; i <= 10; i++ {
        i := i

        wg.Add(1)
        go func() {
            defer wg.Done()

            sem <- struct{}{}
            defer func() {
                <-sem
            }()

            fmt.Println("start:", i)
            time.Sleep(200 * time.Millisecond)
            fmt.Println("done:", i)
        }()
    }

    wg.Wait()
}
```

---

## 【文本块 41】semaphore 和 worker pool 怎么选？

如果任务数量有限，只是想限制同时运行的 goroutine 数量，用 semaphore 很简单。

如果任务是持续流入的，有任务队列、固定 worker、优雅关闭、结果收集、错误处理，worker pool 更合适。

简单说：

* 批量一次性任务：semaphore
* 长期运行任务系统：worker pool
* 需要出错取消和错误聚合：errgroup + context
* 需要复杂队列和监控：成熟任务队列或自研 pool

---

# 十八、并发经典题一：两个 goroutine 交替打印奇偶数

## 【文本块 42】题目：两个 goroutine 交替打印 1 到 10

要求：

* goroutine A 打印奇数
* goroutine B 打印偶数
* 输出顺序必须是 1、2、3、4、...、10

这类题考察的是 goroutine 顺序控制。

可以用 channel 实现两个 goroutine 之间的令牌传递。

oddCh 表示轮到奇数打印。
evenCh 表示轮到偶数打印。
done 表示全部结束。

---

## 【代码块 24】两个 goroutine 交替打印奇偶数

```go
package main

import "fmt"

func main() {
    oddCh := make(chan struct{})
    evenCh := make(chan struct{})
    done := make(chan struct{})

    go func() {
        for i := 1; i <= 9; i += 2 {
            <-oddCh
            fmt.Println(i)
            evenCh <- struct{}{}
        }
    }()

    go func() {
        for i := 2; i <= 10; i += 2 {
            <-evenCh
            fmt.Println(i)

            if i == 10 {
                close(done)
                return
            }

            oddCh <- struct{}{}
        }
    }()

    oddCh <- struct{}{}

    <-done
}
```

---

## 【文本块 43】代码解释

初始时主 goroutine 往 oddCh 发送一个信号，表示先打印奇数。

奇数 goroutine 收到 oddCh 后打印 1，然后通知 evenCh。
偶数 goroutine 收到 evenCh 后打印 2，然后通知 oddCh。
这样两个 goroutine 互相传递令牌，实现严格交替。

最后偶数打印到 10，关闭 done，主 goroutine 退出。

面试里可以这样讲：

这个题的核心是顺序控制。用两个 channel 分别表示奇数和偶数的执行权，每次打印后把执行权交给对方。channel 在这里不是传业务数据，而是传同步信号。

---

# 十九、并发经典题二：三个 goroutine 顺序打印 ABC

## 【文本块 44】题目：三个 goroutine 循环打印 ABC，打印 5 轮

要求输出：

```text
ABCABCABCABCABC
```

可以用三个 channel 控制执行权：

* aCh 控制 A
* bCh 控制 B
* cCh 控制 C

A 打印后通知 B。
B 打印后通知 C。
C 打印后通知 A。
最后一轮 C 打印完后通知 done。

---

## 【代码块 25】三个 goroutine 顺序打印 ABC

```go
package main

import "fmt"

func main() {
    aCh := make(chan struct{})
    bCh := make(chan struct{})
    cCh := make(chan struct{})
    done := make(chan struct{})

    n := 5

    go func() {
        for i := 0; i < n; i++ {
            <-aCh
            fmt.Print("A")
            bCh <- struct{}{}
        }
    }()

    go func() {
        for i := 0; i < n; i++ {
            <-bCh
            fmt.Print("B")
            cCh <- struct{}{}
        }
    }()

    go func() {
        for i := 0; i < n; i++ {
            <-cCh
            fmt.Print("C")

            if i == n-1 {
                close(done)
                return
            }

            aCh <- struct{}{}
        }
    }()

    aCh <- struct{}{}

    <-done
    fmt.Println()
}
```

---

## 【文本块 45】面试表达

这道题本质上是状态机。

A、B、C 三个 goroutine 谁能执行，不是靠抢 CPU，而是靠 channel 传递令牌。

如果要扩展到 N 个 goroutine 顺序执行，也可以用 channel 数组，每个 goroutine 打印后通知下一个。

---

# 二十、并发经典题三：用 Mutex + Cond 打印 ABC

## 【文本块 46】为什么还要会 Cond 版本？

channel 版本更 Go，但 Cond 版本能体现你对条件变量和共享状态的理解。

核心思路：

* 用 state 表示当前轮到谁
* 0 表示 A
* 1 表示 B
* 2 表示 C
* 每个 goroutine 在条件不满足时 Wait
* 打印后修改 state
* Broadcast 唤醒其他等待者

注意 Wait 必须放在 for 里。

---

## 【代码块 26】sync.Cond 顺序打印 ABC

```go
package main

import (
    "fmt"
    "sync"
)

func main() {
    var mu sync.Mutex
    cond := sync.NewCond(&mu)

    state := 0
    n := 5

    printChar := func(target int, char string, next int, wg *sync.WaitGroup) {
        defer wg.Done()

        for i := 0; i < n; i++ {
            mu.Lock()

            for state != target {
                cond.Wait()
            }

            fmt.Print(char)
            state = next

            cond.Broadcast()
            mu.Unlock()
        }
    }

    var wg sync.WaitGroup
    wg.Add(3)

    go printChar(0, "A", 1, &wg)
    go printChar(1, "B", 2, &wg)
    go printChar(2, "C", 0, &wg)

    wg.Wait()
    fmt.Println()
}
```

---

## 【文本块 47】代码解释

state 是共享状态，需要 mu 保护。

每个 goroutine 只有在 state 等于自己目标值时才能打印。
否则调用 cond.Wait 释放锁并等待唤醒。
打印后更新 state，Broadcast 唤醒其他 goroutine。

这里用 Broadcast 而不是 Signal 更简单，因为被唤醒的 goroutine 会重新检查 state，不符合条件会继续 Wait。

---

# 二十一、并发经典题四：生产者消费者

## 【文本块 48】题目：多个生产者、多个消费者处理任务

要求：

* 3 个生产者生产任务
* 2 个消费者消费任务
* 生产者全部结束后关闭任务队列
* 消费者消费完后退出

关键点：

* 生产者不要各自 close jobs
* 等所有生产者结束后，由协调者 close jobs
* 消费者通过 range jobs 自动退出
* WaitGroup 分别管理生产者和消费者

---

## 【代码块 27】多个生产者消费者

```go
package main

import (
    "fmt"
    "sync"
    "time"
)

func producer(id int, jobs chan<- int, wg *sync.WaitGroup) {
    defer wg.Done()

    for i := 1; i <= 3; i++ {
        job := id*10 + i
        jobs <- job
        fmt.Println("producer", id, "send", job)
        time.Sleep(50 * time.Millisecond)
    }
}

func consumer(id int, jobs <-chan int, wg *sync.WaitGroup) {
    defer wg.Done()

    for job := range jobs {
        fmt.Println("consumer", id, "got", job)
        time.Sleep(100 * time.Millisecond)
    }

    fmt.Println("consumer", id, "exit")
}

func main() {
    jobs := make(chan int, 5)

    var producerWG sync.WaitGroup
    var consumerWG sync.WaitGroup

    for i := 1; i <= 3; i++ {
        producerWG.Add(1)
        go producer(i, jobs, &producerWG)
    }

    for i := 1; i <= 2; i++ {
        consumerWG.Add(1)
        go consumer(i, jobs, &consumerWG)
    }

    producerWG.Wait()
    close(jobs)

    consumerWG.Wait()
    fmt.Println("all done")
}
```

---

## 【文本块 49】面试表达

这道题重点不是代码本身，而是关闭权。

多个生产者时，任何一个生产者都不能擅自 close jobs，因为其他生产者可能还在发送。正确做法是用 WaitGroup 等所有生产者结束后，由协调者统一 close jobs。

这能体现你对 channel ownership 的理解：谁负责生产完所有数据，谁负责关闭 channel。

---

# 二十二、并发经典题五：并发执行任务，任一失败就取消全部

## 【文本块 50】题目：多个任务并发执行，只要一个失败，就取消其他任务

这类题在真实项目里非常常见。

比如：

* 并发请求多个下游服务
* 并发拉取多个分片数据
* 并发处理多个文件
* 任一失败就停止整体流程

核心工具：

* context.WithCancel
* error channel
* WaitGroup
* select 监听 ctx.Done

标准库 WaitGroup 不负责错误传播，所以要自己用 errCh，或者实际项目里用 errgroup。

---

## 【代码块 28】手写版本：失败取消全部任务

```go
package main

import (
    "context"
    "errors"
    "fmt"
    "sync"
    "time"
)

func task(ctx context.Context, id int) error {
    select {
    case <-time.After(time.Duration(id) * 100 * time.Millisecond):
        if id == 3 {
            return errors.New("task 3 failed")
        }

        fmt.Println("task success:", id)
        return nil

    case <-ctx.Done():
        fmt.Println("task canceled:", id)
        return ctx.Err()
    }
}

func main() {
    ctx, cancel := context.WithCancel(context.Background())
    defer cancel()

    var wg sync.WaitGroup

    errCh := make(chan error, 1)

    for i := 1; i <= 5; i++ {
        i := i

        wg.Add(1)
        go func() {
            defer wg.Done()

            if err := task(ctx, i); err != nil {
                select {
                case errCh <- err:
                    cancel()
                default:
                }
            }
        }()
    }

    wg.Wait()
    close(errCh)

    if err, ok := <-errCh; ok {
        fmt.Println("first error:", err)
        return
    }

    fmt.Println("all success")
}
```

---

## 【文本块 51】代码解释

errCh 容量是 1，只记录第一个错误。

某个任务失败后：

1. 尝试把错误写入 errCh。
2. 调用 cancel 通知其他任务退出。
3. 其他任务在 select 中监听 ctx.Done。

这里的 select + default 是为了避免多个任务同时失败时阻塞在 errCh 写入上。

实际项目里更推荐使用 errgroup，它封装了这套逻辑。

---

# 二十三、errgroup 简介

## 【文本块 52】Q：errgroup 是什么？和 WaitGroup 有什么区别？

errgroup 是 Go 扩展包里的并发工具，常用于“并发执行多个任务，等待全部完成，并返回第一个错误”。

相比 WaitGroup：

WaitGroup 只等待，不处理错误。
errgroup 可以收集第一个错误。
errgroup.WithContext 可以在一个任务失败时取消其他任务。

典型场景：

* 并发调用多个接口
* 并发查询多个数据源
* 并发执行多个子任务
* 任何一个失败就整体失败

注意：errgroup 在 `golang.org/x/sync/errgroup` 包中，不是标准库内置包。

---

## 【代码块 29】errgroup 示例

```go
package main

import (
    "context"
    "errors"
    "fmt"
    "time"

    "golang.org/x/sync/errgroup"
)

func main() {
    ctx := context.Background()

    g, ctx := errgroup.WithContext(ctx)

    for i := 1; i <= 5; i++ {
        i := i

        g.Go(func() error {
            select {
            case <-time.After(time.Duration(i) * 100 * time.Millisecond):
                if i == 3 {
                    return errors.New("task 3 failed")
                }

                fmt.Println("task success:", i)
                return nil

            case <-ctx.Done():
                fmt.Println("task canceled:", i)
                return ctx.Err()
            }
        })
    }

    if err := g.Wait(); err != nil {
        fmt.Println("group error:", err)
        return
    }

    fmt.Println("all success")
}
```

---

## 【文本块 53】errgroup 工程注意事项

第一，errgroup 只返回第一个错误。
如果你要收集所有错误，需要自己设计 error slice 和锁。

第二，errgroup.WithContext 只有任务返回非 nil error 时才会 cancel。
如果任务内部吞掉错误，其他任务不会被取消。

第三，任务内部要监听 ctx.Done。
否则 cancel 发出了，任务也不会立刻停。

第四，老版本 errgroup 没有并发数限制。
可以配合 semaphore，或者使用支持 SetLimit 的版本。

---

# 二十四、并发安全和 Go 内存模型

## 【文本块 54】Q：什么是数据竞争？

数据竞争是指：

多个 goroutine 同时访问同一变量，至少一个是写操作，并且没有同步保护。

例如：

```go
count++
```

多个 goroutine 同时执行就是数据竞争。

数据竞争会导致结果不可预测，甚至程序行为异常。

Go 提供 race detector，可以用：

```bash
go test -race ./...
```

或者：

```bash
go run -race main.go
```

检测数据竞争。

面试里可以这样回答：

Go 的数据竞争是多个 goroutine 并发访问同一变量，其中至少一个写，而且没有通过锁、channel、atomic 等同步手段建立 happens-before。数据竞争会导致不可预测行为，工程中应该通过 race detector 和代码审查尽早发现。

---

## 【文本块 55】Go 里有哪些同步手段？

常见同步手段：

1. Mutex / RWMutex
2. channel 发送接收
3. close channel
4. WaitGroup
5. Once
6. Cond
7. atomic
8. context 取消信号
9. 外部系统同步，比如数据库事务、Redis 锁

这些同步手段背后都在建立某种 happens-before 关系，让一个 goroutine 的写入对另一个 goroutine 可见。

面试里可以这样回答：

Go 并发安全的核心是避免数据竞争。可以用 Mutex 保护共享状态，用 channel 建立通信同步，用 atomic 做单变量原子操作，用 context 管理生命周期。选择哪种工具取决于问题本质是共享状态保护、任务通信、生命周期取消还是简单状态位更新。

---

# 二十五、本章高频面试题速记

## 【文本块 56】sync 速记

Mutex 用来保护共享状态。
RWMutex 适合读多写少。
Mutex 和 RWMutex 使用后都不能复制。
含锁结构体通常用指针传递。
WaitGroup 用来等待一组 goroutine 完成。
Add 应该在启动 goroutine 前调用。
WaitGroup 只等待，不处理错误和取消。
Once 保证某段逻辑只执行一次，适合单例和初始化。
Once 函数 panic 后也不会重试。
Cond 适合复杂条件等待，Wait 要放在 for 循环里。
Pool 适合复用临时对象，降低 GC 压力，但不能当连接池。
sync.Map 适合读多写少、key 稳定，不是所有并发 map 的默认答案。

---

## 【文本块 57】atomic 速记

atomic 保证单变量操作的原子性和可见性。
常用类型有 atomic.Int64、atomic.Bool、atomic.Value、atomic.Pointer。
Add 适合计数器。
Store/Load 适合状态位。
CompareAndSwap 是 CAS。
atomic.Value 适合读多写少的整体快照。
atomic 不适合复杂业务逻辑和多字段一致性。
高竞争 CAS 自旋可能浪费 CPU。
复杂共享状态优先用 Mutex。

---

## 【文本块 58】context 速记

context 用于取消、超时、deadline 和请求级元数据。
context 通常作为函数第一个参数。
不要传 nil context。
不要把 context 存进 struct。
WithCancel 手动取消。
WithTimeout 相对时间超时。
WithDeadline 绝对时间超时。
派生 context 后要调用 cancel。
context.Value 只传请求级元数据，不要传业务必需参数。
goroutine 要监听 ctx.Done，避免泄漏。
数据库、RPC、HTTP client 都应该传递 ctx。

---

## 【文本块 59】worker pool 速记

Go 不需要为了复用线程而照搬 Java 线程池。
但 Go 需要控制并发度和下游压力。
worker pool 本质是固定数量 worker 消费任务队列。
jobs channel 是任务队列。
results channel 是结果队列。
WaitGroup 等 worker 退出。
context 控制取消。
关闭顺序通常是：关闭 jobs，等待 worker 退出，再关闭 results。
简单批量并发限制可以用 channel semaphore。
需要错误传播和取消时，优先考虑 errgroup + context。
worker 内部要考虑 recover，避免 panic 打崩进程。

---

## 【文本块 60】并发经典题速记

两个 goroutine 交替打印：用两个 channel 传令牌。
三个 goroutine 打印 ABC：用三个 channel 或 Cond + state。
生产者消费者：生产者写 jobs，消费者 range jobs，协调者关闭 jobs。
多个生产者不能各自 close channel，要等全部生产者结束后统一 close。
任一任务失败取消全部：context.WithCancel + errCh + WaitGroup，或 errgroup.WithContext。
channel 解法适合顺序交接。
Cond 解法适合共享状态条件等待。
Mutex 解法适合保护状态变量。

---

# 二十六、本章最终面试回答模板

## 【文本块 61】综合回答模板

如果面试官让我讲 Go 的 sync、atomic、context 和 worker pool，我会这样回答：

Go 的 sync 包主要用于 goroutine 之间的同步和共享状态保护。Mutex 是最基础的互斥锁，用来保护共享变量；RWMutex 适合读多写少场景，允许多个读并发，但写和读、写和写互斥。WaitGroup 用来等待一组 goroutine 完成，但它只负责等待，不负责错误传播和取消。Once 用来做并发安全的一次性初始化。Cond 适合复杂条件等待，Wait 必须放在 for 循环中重新检查条件。Pool 用来复用临时对象，降低 GC 压力，但不能当连接池使用。sync.Map 适合读多写少、key 稳定的场景，不是普通 map 加锁的万能替代品。

atomic 用于单变量的原子读写和更新，比如计数器、开关标志、状态位。CAS 也就是 Compare And Swap，是一种乐观并发控制，比较当前值和预期值，相等才更新。atomic 性能轻量，但只适合简单状态。复杂业务逻辑、多字段一致性、先检查再修改这类场景，还是应该用 Mutex。atomic.Value 常用于读多写少的配置快照，通过构造新对象并整体 Store，实现无锁读取。

context 是 Go 里管理请求生命周期的核心工具。它可以传递取消信号、超时时间、deadline 和请求级元数据。工程里 context 通常作为函数第一个参数，从 HTTP handler 一路传到 service、dao、RPC、DB 查询。派生 context 后要调用 cancel，goroutine 内部要监听 ctx.Done，避免请求结束后 goroutine 还在后台泄漏。context.Value 只适合传 trace_id、user_id 这类请求级元数据，不应该替代显式业务参数。

Go 通常不需要像 Java 那样为了复用线程而设计线程池，因为 goroutine 很轻量，runtime 已经负责线程调度。但工程里仍然需要 worker pool 或 semaphore 来控制并发度，避免下游数据库、Redis、第三方接口被打爆。worker pool 一般由 jobs channel、固定数量 worker、results channel、WaitGroup 和 context 组成。简单批量任务可以用 channel semaphore 控制并发；如果要错误传播和失败取消，实际项目中更推荐 errgroup.WithContext。

并发经典题的核心不是背模板，而是识别同步关系。交替打印奇偶数、ABC 顺序打印，本质是执行权传递，可以用 channel 传令牌，也可以用 Cond + state。生产者消费者要注意 channel ownership，多个生产者时不能各自 close channel，而应该由协调者等所有生产者结束后统一关闭。多个任务并发执行时，如果任一失败就取消全部，可以用 context.WithCancel 配合 error channel，也可以直接使用 errgroup。

一句话总结：Go 并发工程的关键不是无限开 goroutine，而是根据场景选择合适工具。共享状态用 Mutex/RWMutex，简单状态用 atomic，生命周期用 context，任务队列和并发控制用 worker pool 或 semaphore，顺序协作用 channel 或 Cond。
